# Agents War — starter notebook

**Google Colab:** open this notebook from the GitHub repo in Colab
(File → Open notebook → GitHub tab → paste the repo link), then run the cells top to bottom (Shift + Enter).
The first cell downloads the rest of the kit for you.

**Laptop:** `git clone` the repo (or Code → Download ZIP), open this notebook inside that folder and run it.

⚠️ Colab deletes your files when the session ends. Run the **last cell** often to download `strategy.py`.
That file is what you submit.

In [ ]:
# ---- Setup (run once) ----
REPO_URL = "https://github.com/0770-Lakshya/ALPHA-ARENA-Agents-War-.git"
import os, subprocess, importlib
if not os.path.exists("engine.py"):
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(folder):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(folder)

import engine
data = engine.load("data/train.csv")
print("Kit ready:", len(data["close"]), "days x", data["close"].shape[1], "stocks")
data["close"].tail()

## Your strategies
**This cell is your `strategy.py`.** Edit it and run it: the first line saves it to the file.
Set `TRAIN_WINDOW` (any block of max 10 years = 2520 days), learn in `train()`,
write each idea as its own `decide_xxx` function and point `decide` at your final one.
(Use either this notebook or edit `strategy.py` directly, not both: running this cell overwrites the file.)

In [ ]:
%%writefile strategy.py
"""
AGENTS WAR - your trading agent.  This is the ONLY code file you submit.

How to work:
  1. Choose TRAIN_WINDOW: the block of days your agent learns from. It can be ANY block of
     at most 10 years (2520 days) inside days 0..3707, e.g. (0, 2519), (252, 2771), (1188, 3707).
  2. train(data) is called ONCE with only those days. Learn your parameters / fit your models there.
  3. Write every trading idea as its own function named  decide_<something>(history).
     decide only sees the last year of prices, so all long-term learning must happen in train().
  4. python compare.py          -> tests all your decide_ ideas side by side
     python compare.py windows  -> tests your final agent trained on 6 different 10-year windows
  5. Point  decide  (bottom of this file) at the one you want us to run on the secret period.

Rules for this file:
  * TRAIN_WINDOW, train (optional) and decide must exist with these names.
  * A TRAIN_WINDOW longer than 2520 days is rejected (disqualified).
  * Allowed imports: python standard library, numpy, pandas, scikit-learn.
  * No internet, no reading other files, no hard-coded prices or day numbers in decide,
    and no collecting prices day by day inside decide to get around the 1-year limit.
  * On average your agent must keep at least 30% of the money invested.
  * A full run (training + secret days) must finish within 3 minutes.
"""
import numpy as np
import pandas as pd

TEAM_NAME = "<Your team name here>"
TRAIN_WINDOW = (0, 2519)      # (first_day, last_day) your agent learns from - at most 2520 days
MODEL = {}                    # whatever train() learns is stored here

# data / history: dict with keys "open", "high", "low", "close", "volume".
#          Each one is a pandas DataFrame: rows = days (last row = today),
#          columns = stocks (STOCK_01 ... STOCK_12).
# decide returns: dict {stock: weight}
#          weight =  0.25 -> put 25% of your money in that stock (buy)
#          weight = -0.10 -> bet 10% against that stock (short sell)
#          stocks you leave out get 0; whatever is unused stays in cash.
#          The sum of |weights| must be <= 1 (scaled down for you if it is more).


def train(data):
    """Called once, with ONLY the days in TRAIN_WINDOW.
    Example: find which moving-average length worked best inside the training window."""
    close = data["close"]
    next_day = close.pct_change().shift(-1)
    best_sharpe, best_days = -np.inf, 50
    for days in [20, 50, 100, 200]:
        hold = close > close.rolling(days).mean()                  # stocks above their average
        daily = (next_day * hold).sum(axis=1) / hold.sum(axis=1).replace(0, np.nan)
        sharpe = daily.mean() / daily.std()
        if sharpe > best_sharpe:
            best_sharpe, best_days = sharpe, days
    MODEL["ma_days"] = best_days


def decide_momentum(history):
    """Buy the 3 stocks that went up the most in the last 20 days."""
    close = history["close"]
    if len(close) < 21:
        return {}                                   # not enough data yet -> stay in cash
    past_return = close.iloc[-1] / close.iloc[-21] - 1
    winners = past_return.nlargest(3).index
    return {stock: 1 / 3 for stock in winners}


def decide_trend_filter(history):
    """Hold every stock that is above its moving average (length learnt in train), equally weighted."""
    close = history["close"]
    days = MODEL.get("ma_days", 50)
    if len(close) < days:
        return {}
    above = close.iloc[-1] > close.iloc[-days:].mean()
    picks = above[above].index
    return {stock: 1 / len(picks) for stock in picks} if len(picks) else {}


# Ideas to try next: mean reversion, RSI, volatility filters, stop-losses,
# position sizing by risk, ML models fitted in train(), ...


decide = decide_trend_filter      # <- the agent we will run on the secret period


if __name__ == "__main__":
    # python strategy.py -> trains on TRAIN_WINDOW and prints the performance metrics.
    # (Only runs when you start this file yourself; our evaluation just imports it.)
    import sys
    from pathlib import Path
    import engine

    data = engine.load(Path(__file__).resolve().parent / "data" / "train.csv")
    window = engine.train_agent(sys.modules[__name__], data)
    log = engine.backtest(decide, data, verbose=False)["log"]
    inside = (log.index >= window[0]) & (log.index <= window[1])
    print(f"{TEAM_NAME}: trained on days {window[0]}-{window[1]}")
    print("\nInside the training window:")
    engine.print_metrics(engine.metrics(log[inside]))
    print("Outside the training window (validation):")
    engine.print_metrics(engine.metrics(log[~inside]))

## Compare all your strategies
Trains on your `TRAIN_WINDOW`, then tests every `decide_xxx` on the days **outside** the window (validation).
It saves `comparison.csv` for your slides.

In [ ]:
import strategy, compare
importlib.reload(strategy); importlib.reload(compare)
compare.main()

## Try different 10-year training windows
Trains your final `decide` on 6 different 10-year windows and tests each one outside its window. It saves `windows.csv`.

In [ ]:
compare.main("windows")

## Performance metrics of your chosen agent (same as `python strategy.py`)

In [ ]:
import runpy
_ = runpy.run_path("strategy.py", run_name="__main__")

## Your visualization (deliverable)
**This cell is your `visualization.py`.** Write your portfolio growth graph here and run it:
X-axis = Day, Y-axis = portfolio value in Rs starting from Rs 1,00,000, at most 10 years, with the benchmark line.

In [ ]:
%%writefile visualization.py
"""
visualization.py - YOU write this file. It is one of your deliverables.

Make a cumulative portfolio growth graph of your strategy:
  * X-axis: Day
  * Y-axis: Portfolio value in Rs, starting from Rs 1,00,000
  * At most 10 years (2520 days) on one graph, e.g. your training window or your validation days
  * Add the equal-weight benchmark (engine.equal_weight) as a second line so we can compare

Useful pieces:
  data = engine.load("data/train.csv")
  engine.train_agent(strategy, data)                    # trains your agent on TRAIN_WINDOW
  result = engine.backtest(strategy.decide, data, verbose=False)
  result["log"]["return"]                               # your daily returns, index = day
  engine.STARTING_CAPITAL                               # Rs 1,00,000

Run it:   python visualization.py
"""
import matplotlib.pyplot as plt

import engine
import strategy

# TODO: your code here

In [ ]:
_ = runpy.run_path("visualization.py", run_name="__main__")

## Download your submission files

In [ ]:
try:
    from google.colab import files
    for extra in ["strategy.py", "visualization.py", "comparison.csv", "windows.csv"]:
        if os.path.exists(extra):
            files.download(extra)
except ImportError:
    print("Not on Colab - your files are in:", os.getcwd())